# All-Analog GPT-2 on Heterogeneous 3D-CIM — Phase Selector

This notebook uses **only code committed to**:

`https://github.com/williamhwangweiju/projection-sensitivity-mapping`

The setup fetches the selected branch, resets the Colab checkout to that
commit, and invokes the phase functions directly from the repository.

Pipeline order: **Phase 0 (HWA fine-tune) → Phase 1 (sensitivity profiling) → Phase 2 (fidelity trace) → Phase 3 (placement) → validation →
Phase 4 (all-analog quality)**.

Set `USE_HWA = False` for the vanilla post-training (PTQ) contrast — the
same pipeline on pretrained weights.


In [ ]:
#@title 1. Experiment and phase settings
from pathlib import Path

REPO_URL = "https://github.com/williamhwangweiju/projection-sensitivity-mapping.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

RUN_MODE = "full"  #@param ["smoke", "full"]
SEED = 42  #@param {type:"integer"}

# Hardware-aware deployment: Phase 0 fine-tunes under the deployment noise
# model and every later phase loads that checkpoint. Set False for the PTQ
# contrast (pretrained weights, no Phase 0).
USE_HWA = True  #@param {type:"boolean"}

# Set a phase to 1 to run it, or 0 to skip it. Enabled phases always execute
# in pipeline order: Phase 0 -> 1 -> 2 -> 3 -> 4.
RUN_PHASE0 = 0  #@param {type:"integer"}
RUN_PHASE1 = 0  #@param {type:"integer"}
RUN_PHASE2 = 0  #@param {type:"integer"}
RUN_PHASE3 = 0  #@param {type:"integer"}
RUN_PHASE4 = 0  #@param {type:"integer"}

MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}
PREFER_GPU = True  #@param {type:"boolean"}
RUN_TESTS = False  #@param {type:"boolean"}

PHASE_FLAGS = {
    "phase0": RUN_PHASE0 if USE_HWA else 0,
    "phase1": RUN_PHASE1,
    "phase2": RUN_PHASE2,
    "phase3": RUN_PHASE3,
    "phase4": RUN_PHASE4,
}
invalid_phase_flags = {
    name: value
    for name, value in PHASE_FLAGS.items()
    if value not in (0, 1)
}
if invalid_phase_flags:
    raise ValueError(
        "Every phase flag must be exactly 0 or 1: "
        f"{invalid_phase_flags}"
    )

PHASES_TO_RUN = [name for name, enabled in PHASE_FLAGS.items() if enabled == 1]

# Leave blank to use the newest matching artifact in Drive.
PHASE1_ARTIFACT = ""  #@param {type:"string"}
PHASE2_TRACE_ARTIFACT = ""  #@param {type:"string"}
PHASE3_MANIFEST_ARTIFACT = ""  #@param {type:"string"}

PROJECT_DIR = Path("/content/projection-sensitivity-mapping")
IBM_3DSIM_DIR = PROJECT_DIR / "simulators" / "ibm_3d_cim"
IBM_3DSIM_URL = "https://github.com/IBM/3D-CiM-LLM-Inference-Simulator.git"

DRIVE_ROOT = Path("/content/drive/MyDrive/projection-sensitivity-mapping-hybrid-auto")
LOCAL_ROOT = Path("/content/projection-sensitivity-mapping-hybrid-auto")


In [ ]:
#@title 2. Mount Drive and clone/reset the GitHub repository
import os
import shutil
import subprocess

def run_checked(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = DRIVE_ROOT
else:
    STORAGE_ROOT = LOCAL_ROOT

MODEL_VARIANT = "hwa" if USE_HWA else "ptq"
RESULTS_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / f"seed_{SEED}"
LOG_ROOT = STORAGE_ROOT / "logs"
CACHE_ROOT = STORAGE_ROOT / "cache"
CONFIG_ROOT = STORAGE_ROOT / "configs"

for path in (STORAGE_ROOT, RESULTS_ROOT, LOG_ROOT, CACHE_ROOT, CONFIG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

# Keep hot Hugging Face caches on local disk: heavy cache I/O through the
# Drive FUSE mount is a common cause of "Transport endpoint is not
# connected" mount failures mid-run. Downloads repeat per session
# (wikitext + gpt2, a few minutes) but results on Drive are unaffected.
LOCAL_CACHE = Path("/content/hf_cache")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(LOCAL_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(LOCAL_CACHE / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if (PROJECT_DIR / ".git").is_dir():
    run_checked(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run_checked(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=PROJECT_DIR)
    run_checked(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=PROJECT_DIR)
    run_checked(["git", "clean", "-fd"], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_checked([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "--recurse-submodules", "--shallow-submodules",
        REPO_URL, PROJECT_DIR,
    ])

run_checked(["git", "submodule", "sync", "--recursive"], cwd=PROJECT_DIR)
run_checked(["git", "submodule", "update", "--init", "--recursive"], cwd=PROJECT_DIR)

if not (IBM_3DSIM_DIR / "setup.py").is_file():
    if IBM_3DSIM_DIR.exists():
        shutil.rmtree(IBM_3DSIM_DIR)
    IBM_3DSIM_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_checked(["git", "clone", "--depth", "1", IBM_3DSIM_URL, IBM_3DSIM_DIR])

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True
).strip()
STATUS = subprocess.check_output(
    ["git", "status", "--short"], cwd=PROJECT_DIR, text=True
).strip()

print("Repository:", PROJECT_DIR)
print("Branch:", BRANCH)
print("Commit:", COMMIT)
print("Working tree clean:", STATUS == "")
print("Results root:", RESULTS_ROOT)

In [ ]:
#@title 3. Install dependencies, AIHWKit, and IBM 3D-CIM
import platform
import subprocess
import sys
from pathlib import Path


def shell(command: str) -> None:
    print("+", command)
    subprocess.run(["bash", "-lc", command], check=True)


shell("apt-get update -qq")
shell(
    "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    "libopenblas-dev build-essential cmake ninja-build graphviz"
)
shell(f"{sys.executable} -m pip install -q --upgrade pip setuptools wheel")

python_dependencies = [
    "transformers>=4.30,<5",
    "datasets>=2.14,<4",
    "pandas>=2.2,<3",
    "PyYAML>=6",
    "matplotlib>=3.7",
    "tqdm>=4.65",
    "pytest>=7",
    "pydot>=1.4",
]
shell(
    f"{sys.executable} -m pip install -q --upgrade --no-cache-dir "
    + " ".join(f"'{dependency}'" for dependency in python_dependencies)
)

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
has_gpu_runtime = subprocess.run(
    ["bash", "-lc", "command -v nvidia-smi >/dev/null 2>&1"],
    check=False,
).returncode == 0

if PREFER_GPU and has_gpu_runtime and py_tag in {"cp310", "cp311", "cp312"}:
    wheel_name = (
        f"aihwkit-1.1.0-{py_tag}-{py_tag}-"
        "manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
    )
    wheel_url = (
        "https://aihwkit-gpu-demo.s3.us-east.cloud-object-storage.appdomain.cloud/"
        + wheel_name
    )
    wheel_path = Path("/content") / wheel_name
    shell(f"wget -q --show-progress -O '{wheel_path}' '{wheel_url}'")
    shell(
        f"{sys.executable} -m pip install -q "
        f"--force-reinstall --no-deps '{wheel_path}'"
    )
    AIHWKIT_INSTALL_KIND = "official GPU wheel"
else:
    shell(
        f"{sys.executable} -m pip install -q "
        "--upgrade --no-cache-dir 'aihwkit==1.1.0'"
    )
    AIHWKIT_INSTALL_KIND = "PyPI CPU package"

shell(
    f"{sys.executable} -m pip install -q "
    f"--no-deps -e '{IBM_3DSIM_DIR}'"
)

print("Python:", platform.python_version())
print("AIHWKit installation:", AIHWKIT_INSTALL_KIND)
print("IBM 3D-CIM editable package:", IBM_3DSIM_DIR)

In [ ]:
#@title 4. Verify AIHWKit, threedsim, and CUDA
import importlib
import importlib.metadata
import os
import subprocess
import sys

import torch
import transformers
import datasets
import aihwkit
from aihwkit.nn import AnalogLinear

IBM_3DSIM_SRC = IBM_3DSIM_DIR / "src"
simulator_src = str(IBM_3DSIM_SRC)

if simulator_src not in sys.path:
    sys.path.insert(0, simulator_src)

pythonpath_parts = [str(PROJECT_DIR), simulator_src]
existing_pythonpath = os.environ.get("PYTHONPATH", "")
if existing_pythonpath:
    pythonpath_parts.append(existing_pythonpath)
os.environ["PYTHONPATH"] = os.pathsep.join(pythonpath_parts)

importlib.invalidate_caches()
import threedsim

RUNTIME_DEVICE = "cuda" if PREFER_GPU and torch.cuda.is_available() else "cpu"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Probe the RPUCuda MAPPED-tile path the pipeline actually uses, with the
# repository's own RPU config. Plain AnalogLinear can succeed while mapped
# RPUCuda tiles fail CUBLAS initialization (e.g. a GPU architecture the
# prebuilt aihwkit wheel was not compiled for), which would otherwise only
# surface hours later inside Phase 1 or Phase 4.
from aihwkit.nn import AnalogLinearMapped
from src.common.analog import ManualAnalogSettings, make_rpu_config

_probe_settings = ManualAnalogSettings(
    clip_sigma=2.5, range_mode="peak_to_peak", reference_noise_std=0.023,
    tile_size=512, adc_dac_bits=8, output_bound=None,
    weight_scaling_omega=1.0, weight_scaling_columnwise=False,
)
try:
    layer = AnalogLinearMapped(
        768, 1024, bias=True, rpu_config=make_rpu_config(_probe_settings)
    )
    if RUNTIME_DEVICE == "cuda":
        layer = layer.cuda()
        probe_input = torch.zeros(1, 768, device="cuda")
    else:
        probe_input = torch.zeros(1, 768)
    probe_output = layer(probe_input)
    if RUNTIME_DEVICE == "cuda":
        torch.cuda.synchronize()
    del layer
    if RUNTIME_DEVICE == "cuda":
        torch.cuda.empty_cache()
except Exception as exc:
    print("=" * 72)
    print("AIHWKit MAPPED-TILE GPU PROBE FAILED:", repr(exc))
    print("This GPU cannot run the pipeline's analog tiles (likely a GPU")
    print("architecture unsupported by the prebuilt aihwkit wheel).")
    print("Falling back to CPU would make Phase 1/4 take DAYS. Instead:")
    print("Runtime -> Change runtime type -> pick a different GPU (e.g. T4),")
    print("then rerun from Cell 2. Check the GPU with: !nvidia-smi -L")
    print("=" * 72)
    RUNTIME_DEVICE = "cpu"
    probe_output = AnalogLinear(2, 2)(torch.tensor([[0.1, 0.2]]))

subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import threedsim; "
            "from threedsim.accelerator import Accelerator, AcceleratorConfig; "
            "from threedsim.mapping import Mapper, MapStrategy, Strategy; "
            "print('Subprocess threedsim:', threedsim.__file__)"
        ),
    ],
    env=os.environ.copy(),
    check=True,
)

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "selected_device": RUNTIME_DEVICE,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "aihwkit": getattr(
        aihwkit, "__version__", importlib.metadata.version("aihwkit")
    ),
    "threedsim": threedsim.__file__,
})
print("AIHWKit probe:", probe_output.detach().cpu().numpy())

In [ ]:
#@title 5. Edit and create the complete Drive-backed experiment config
import yaml

full_config = PROJECT_DIR / "configs/full_pipeline/gpt2_hybrid_3dcim.yaml"
smoke_config = PROJECT_DIR / "configs/full_pipeline/gpt2_hybrid_3dcim_smoke.yaml"

BASE_CONFIG = full_config if RUN_MODE == "full" else smoke_config
if not BASE_CONFIG.is_file():
    raise FileNotFoundError(
        "The selected hybrid config is not present in the GitHub checkout: "
        f"{BASE_CONFIG}"
    )

# The repository configuration is the source of truth. Put ONLY the values
# you want to change in this block; it is deep-merged over the repository
# config, so every repository key you do not override (hwa_training, policy
# lists, ...) stays active.
# NOTE: lists replace rather than merge — overriding e.g. phase4.policies
# here replaces the full repository list.
COLAB_CONFIG_OVERRIDES_YAML = r"""
{}
# --- examples ---
# hwa_training:
#   max_steps: 2000
#   precision: bf16          # use fp32 on a T4
#   dataset:
#     batch_size: 8          # lower this if the GPU runs out of memory
# profiling:
#   num_seeds: 5
# phase4:
#   num_realizations: 3
"""


def deep_merge(base, overrides):
    """Recursively merge Colab overrides without discarding new base keys."""
    for key, value in overrides.items():
        if (
            key in base
            and isinstance(base[key], dict)
            and isinstance(value, dict)
        ):
            deep_merge(base[key], value)
        else:
            base[key] = value
    return base


config = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
overrides = yaml.safe_load(COLAB_CONFIG_OVERRIDES_YAML)
if not isinstance(config, dict):
    raise ValueError("Repository config must contain a YAML mapping.")
if overrides is None:
    overrides = {}
if not isinstance(overrides, dict):
    raise ValueError("COLAB_CONFIG_OVERRIDES_YAML must contain a YAML mapping.")
config = deep_merge(config, overrides)

config.setdefault("experiment", {})
config["experiment"]["seed"] = int(SEED)
config["experiment"]["placement_seed"] = int(SEED)
config.setdefault("model", {})
config["model"]["device"] = RUNTIME_DEVICE

phase_output_roots = {
    "hwa_training": RESULTS_ROOT / "phase0",
    "phase1": RESULTS_ROOT / "phase1",
    "digital_selection": RESULTS_ROOT / "phase1_5_digital_selection",
    "phase2": RESULTS_ROOT / "phase2",
    "phase3": RESULTS_ROOT / "phase3",
    "phase4": RESULTS_ROOT / "phase4",
}
for section, output_root in phase_output_roots.items():
    if section in config and isinstance(config[section], dict):
        config[section]["output_root"] = str(output_root)

# Keep the Phase 0 contract consistent: model.checkpoint must equal
# <hwa_training.output_root>/checkpoint_final when HWA is used, and must be
# null for the PTQ contrast.
# The calibration tree normally lives in RESULTS_ROOT (seed_42/); after the
# Drive reorganization it may live inside multiseed/trace_seed_<SEED>/ instead.
# Accept either layout.
TRACE42_ROOT = (
    STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT
    / "multiseed" / f"trace_seed_{SEED}"
)
_checkpoint_candidates = [
    RESULTS_ROOT / "phase0" / "checkpoint_final",
    TRACE42_ROOT / "phase0" / "checkpoint_final",
]
HWA_CHECKPOINT_DIR = next(
    (path for path in _checkpoint_candidates if path.is_dir()),
    _checkpoint_candidates[0],
)
if USE_HWA:
    config.setdefault("hwa_training", {})["enabled"] = True
    config["model"]["checkpoint"] = str(HWA_CHECKPOINT_DIR)
else:
    if "hwa_training" in config:
        config["hwa_training"]["enabled"] = False
    config["model"]["checkpoint"] = None

COLAB_CONFIG = CONFIG_ROOT / (
    f"gpt2_hybrid_{RUN_MODE}_{'hwa' if USE_HWA else 'ptq'}_seed{SEED}.yaml"
)
COLAB_CONFIG.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)

print("Repository base config:", BASE_CONFIG)
print("Generated Drive config:", COLAB_CONFIG)
print("Selected phases:", PHASES_TO_RUN)
print("HWA mode:", USE_HWA, "| checkpoint:", config["model"]["checkpoint"])
print("Phase 4 policies:", config.get("phase4", {}).get("policies"))
print(
    "Selection methods:",
    config.get("digital_selection", {}).get("methods"),
)


In [ ]:
#@title 6. Optional repository tests
import os
import subprocess
import sys

if RUN_TESTS:
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=PROJECT_DIR,
        env=os.environ.copy(),
        check=True,
    )
    smoke_script = PROJECT_DIR / "scripts/smoke_aihwkit_contract.py"
    if smoke_script.is_file():
        subprocess.run(
            [sys.executable, str(smoke_script), "--device", RUNTIME_DEVICE],
            cwd=PROJECT_DIR,
            env=os.environ.copy(),
            check=True,
        )
    print("Repository tests passed.")
else:
    print("Tests skipped.")

In [ ]:
#@title 7. Resolve existing artifacts from Google Drive
from pathlib import Path

# Artifacts may live in the seed tree (original layout) or under
# multiseed/trace_seed_<SEED> (reorganized layout). Search both.
ARTIFACT_SEARCH_ROOTS = [RESULTS_ROOT, TRACE42_ROOT]

def newest(patterns, *, label, required=False):
    matches = []
    for root in ARTIFACT_SEARCH_ROOTS:
        for pattern in patterns:
            matches.extend(root.glob(pattern))
    matches = [path for path in matches if path.is_file()]
    matches.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(
            f"No {label} artifact found under {RESULTS_ROOT}. "
            f"Patterns: {patterns}"
        )
    return None

def explicit_or_latest(raw_path, patterns, label, required=False):
    if str(raw_path).strip():
        path = Path(str(raw_path).strip())
        if not path.is_file():
            raise FileNotFoundError(f"{label} does not exist: {path}")
        return path
    return newest(patterns, label=label, required=required)

PHASE1_PATH = explicit_or_latest(
    PHASE1_ARTIFACT,
    ["phase1/*sensitivity*.json", "phase1/**/*sensitivity*.json"],
    "Phase 1 profile",
)
# proxy_sensitivity files also match *sensitivity*; exclude them from Phase 1.
if PHASE1_PATH is not None and PHASE1_PATH.name.startswith("proxy_sensitivity"):
    candidates = sorted(
        (
            path
            for root in ARTIFACT_SEARCH_ROOTS
            for path in root.glob("phase1/*sensitivity*.json")
            if not path.name.startswith("proxy_sensitivity")
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    PHASE1_PATH = candidates[0] if candidates else None
TRACE_PATH = explicit_or_latest(
    PHASE2_TRACE_ARTIFACT,
    ["phase2/**/trace.npz"],
    "Phase 2 trace",
)
PHASE3_MANIFEST_PATH = explicit_or_latest(
    PHASE3_MANIFEST_ARTIFACT,
    ["phase3/phase3_manifest.json", "phase3/**/phase3_manifest.json"],
    "Phase 3 manifest",
)
print("Existing artifacts:")
print("Phase 1:", PHASE1_PATH)
print("Phase 2:", TRACE_PATH)
print("Phase 3:", PHASE3_MANIFEST_PATH)
if USE_HWA:
    print(
        "Phase 0 checkpoint present:",
        HWA_CHECKPOINT_DIR.is_dir(),
        "|",
        HWA_CHECKPOINT_DIR,
    )

# The executable multi-phase runner is defined immediately below.

RUNNER_SOURCE = r'''
import json
import os
import sys
from pathlib import Path

repo = Path(os.environ["PSM_PROJECT_DIR"])
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

actions = json.loads(os.environ["PSM_ACTIONS"])
config = Path(os.environ["PSM_CONFIG"])


def optional(name):
    value = os.environ.get(name, "").strip()
    return None if not value else Path(value)


phase1 = optional("PSM_PHASE1")
trace = optional("PSM_TRACE")
phase3 = optional("PSM_PHASE3")


def require(path, label):
    if path is None or not path.is_file():
        raise FileNotFoundError(
            f"{label} artifact is required but was not produced in this run "
            f"and was not found in Drive: {path}"
        )
    return path


def run_phase0():
    from experiments.phase0_hwa_training.run_hwa_training import main
    return main(config)


def run_phase1():
    from experiments.phase1_sensitivity.run_aihwkit_profiling import main
    from experiments.phase1_sensitivity.analyze_results import main as analyze
    result = main(config)
    analyze(result)
    return result


def run_phase2():
    from experiments.phase2_fidelity.run_fidelity_model import main
    return main(config)


def run_phase3(phase1_path, trace_path):
    from experiments.phase3_baselines.run_baseline_mappings import main
    return main(config, phase1_path, trace_path)


def validate(phase1_path, trace_path, phase3_path):
    from scripts.validate_pipeline_contracts import validate_pipeline
    validate_pipeline(config, phase1_path, trace_path, phase3_path)


def run_phase4(phase1_path, trace_path, phase3_path):
    from experiments.phase4_quality.run_hybrid_quality import main
    return main(config, phase1_path, trace_path, phase3_path)


for action in actions:
    print(f"\n===== Running {action} =====")
    if action == "phase0":
        checkpoint = run_phase0()
        print("Phase 0 checkpoint:", checkpoint)
    elif action == "phase1":
        phase1 = run_phase1()
        print("Phase 1:", phase1)
    elif action == "phase2":
        trace = run_phase2()
        print("Phase 2:", trace)
    elif action == "phase3":
        phase3 = run_phase3(
            require(phase1, "Phase 1"),
            require(trace, "Phase 2"),
        )
        print("Phase 3:", phase3)
    elif action == "phase4":
        phase1 = require(phase1, "Phase 1")
        trace = require(trace, "Phase 2")
        phase3 = require(phase3, "Phase 3")
        validate(phase1, trace, phase3)
        phase4 = run_phase4(phase1, trace, phase3)
        print("Phase 4:", phase4)
    else:
        raise ValueError(f"Unsupported phase: {action}")

print(f"\nSelected phases complete: {actions}")
'''


In [ ]:
#@title 8. Run the selected phases
from datetime import datetime
import json
import os
import subprocess
import sys

env = os.environ.copy()
env.update({
    "PYTHONUNBUFFERED": "1",
    "HF_HOME": os.environ["HF_HOME"],
    "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONPATH": os.environ["PYTHONPATH"],
})

phase_label = "_".join(PHASES_TO_RUN)
log_path = LOG_ROOT / (
    f"{phase_label}_{RUN_MODE}_seed{SEED}_"
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
)

run_env = env.copy()
run_env.update({
    "PSM_PROJECT_DIR": str(PROJECT_DIR),
    "PSM_ACTIONS": json.dumps(PHASES_TO_RUN),
    "PSM_CONFIG": str(COLAB_CONFIG),
    "PSM_PHASE1": "" if PHASE1_PATH is None else str(PHASE1_PATH),
    "PSM_TRACE": "" if TRACE_PATH is None else str(TRACE_PATH),
    "PSM_PHASE3": "" if PHASE3_MANIFEST_PATH is None else str(PHASE3_MANIFEST_PATH),
})

command = [sys.executable, "-u", "-c", RUNNER_SOURCE]
print("Phases:", PHASES_TO_RUN)
print("Configuration:", COLAB_CONFIG)
print("Log:", log_path)

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Selected phases failed with exit code {return_code}. "
        f"The complete traceback is above and in {log_path}."
    )

print("Completed phases:", PHASES_TO_RUN)
print("Results:", RESULTS_ROOT)
print("Log:", log_path)


In [ ]:
#@title 8b. Run one hardware-trace seed (reuses Phase 0/1 from the SEED tree)
#@markdown Keep Cell 1 at `SEED = 42` and all `RUN_PHASE... = 0`, run Cells 1-7,
#@markdown then run this cell once per session with a new `TRACE_SEED`.
#@markdown Rerunning the same command after a preemption resumes Phase 4 from
#@markdown its per-timestep checkpoint.
TRACE_SEED = 45  #@param {type:"integer"}

import subprocess, sys
MULTISEED_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "multiseed"
command = [
    sys.executable, "-u", "scripts/run_multiseed_pipeline.py",
    "--config", str(COLAB_CONFIG),
    "--phase1", str(PHASE1_PATH),
    "--trace-seeds", str(TRACE_SEED),
    "--vary-placement-seed",
    "--output-root", str(MULTISEED_ROOT),
]
print("+", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)
print("Trace seed", TRACE_SEED, "complete. Manifest:", MULTISEED_ROOT / "multiseed_run_manifest.yaml")


In [ ]:
#@title 8c. Aggregate across trace seeds (run after the last seed)
#@markdown Adds the original single-seed run (SEED tree) to the manifest if it
#@markdown is missing, then produces the cross-trace paper table.
import subprocess, sys, yaml
from pathlib import Path

MULTISEED_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "multiseed"
manifest_path = MULTISEED_ROOT / "multiseed_run_manifest.yaml"
payload = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
runs = payload.get("runs", [])

# The seed-42 run may live in the seed tree (original layout) or under
# multiseed/trace_seed_<SEED> (reorganized layout). Resolve whichever exists
# and self-heal a stale entry left by the other layout.
_seed_run_roots = [RESULTS_ROOT, TRACE42_ROOT]
seed_run_root = next(
    (root for root in _seed_run_roots
     if (root / "phase4" / "phase4_metadata.json").is_file()),
    None,
)
if seed_run_root is None:
    raise FileNotFoundError(
        f"No phase4/phase4_metadata.json for trace seed {SEED} under "
        f"{[str(root) for root in _seed_run_roots]}"
    )
seed_entry = {
    "trace_seed": int(SEED),
    "runtime_config": str(COLAB_CONFIG),
    "output_root": str(seed_run_root),
    "phase4_metadata": str(seed_run_root / "phase4/phase4_metadata.json"),
}
changed = False
existing = [run for run in runs if int(run["trace_seed"]) == int(SEED)]
if not existing:
    runs.append(seed_entry)
    changed = True
    print(f"Added trace seed {SEED} to the manifest.")
elif not Path(str(existing[0]["phase4_metadata"])).is_file():
    existing[0].update(seed_entry)
    changed = True
    print(f"Healed stale manifest entry for trace seed {SEED}.")
if changed:
    payload["runs"] = sorted(runs, key=lambda run: int(run["trace_seed"]))
    manifest_path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")

output_csv = MULTISEED_ROOT / "cross_trace_paired_summary.csv"
subprocess.run(
    [sys.executable, "scripts/aggregate_multiseed.py",
     "--manifest", str(manifest_path), "--output", str(output_csv)],
    cwd=PROJECT_DIR, check=True,
)
import pandas as pd
from IPython.display import display
display(pd.read_csv(output_csv))


In [ ]:
#@title 8d. Bounds: adversarial + clairvoyant oracle (one trace seed per run)
#@markdown Additive experiment on top of the frozen five-policy campaign. Run
#@markdown Cells 1-7 first (SEED = 42, every RUN_PHASE... = 0), then run this
#@markdown cell once per `BOUNDS_TRACE_SEED` in 41..45. Phase 0/1 artifacts are
#@markdown reused; each seed's Phase-2 trace is regenerated deterministically
#@markdown (Cell 8e verifies bit-parity against the frozen trace). Phase-4
#@markdown noise fields are keyed by (seed, projection, realization) only, so
#@markdown these rows stay exactly paired with the existing results. Rerunning
#@markdown after a preemption resumes Phase 4 from its checkpoint.
BOUNDS_TRACE_SEED = 41  #@param {type:"integer"}

import subprocess, sys, yaml

assert USE_HWA, "The bounds experiment extends the HWA study; set USE_HWA = True."

bounds_base = PROJECT_DIR / "configs/full_pipeline/gpt2_allanalog_bounds.yaml"
bounds_config = yaml.safe_load(bounds_base.read_text(encoding="utf-8"))
# Same Drive fixups as Cell 5. Phase 2/3/4 output roots are redirected per
# trace seed by run_multiseed_pipeline.py; phase2 parameters are untouched so
# the regenerated traces match the frozen campaign exactly.
bounds_config.setdefault("experiment", {})
bounds_config["experiment"]["seed"] = int(SEED)
bounds_config["experiment"]["placement_seed"] = int(SEED)
bounds_config.setdefault("model", {})
bounds_config["model"]["device"] = RUNTIME_DEVICE
bounds_config["model"]["checkpoint"] = str(HWA_CHECKPOINT_DIR)

BOUNDS_CONFIG = CONFIG_ROOT / f"gpt2_bounds_{RUN_MODE}_{MODEL_VARIANT}.yaml"
BOUNDS_CONFIG.write_text(yaml.safe_dump(bounds_config, sort_keys=False), encoding="utf-8")
BOUNDS_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "multiseed_bounds"

command = [
    sys.executable, "-u", "scripts/run_multiseed_pipeline.py",
    "--config", str(BOUNDS_CONFIG),
    "--phase1", str(PHASE1_PATH),
    "--trace-seeds", str(BOUNDS_TRACE_SEED),
    "--vary-placement-seed",
    "--output-root", str(BOUNDS_ROOT),
]
print("+", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)
print("Bounds run complete for trace seed", BOUNDS_TRACE_SEED)


In [ ]:
#@title 8e. Bounds: verify trace parity and build the span table
#@markdown Asserts every regenerated bounds trace is identical to the frozen
#@markdown campaign trace, merges the bounds rows with the frozen five-policy
#@markdown rows, and reports the placement span (adversarial -> oracle) plus
#@markdown the fraction of headroom static_sensitivity captures.
import numpy as np
import pandas as pd
from IPython.display import display

BOUNDS_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "multiseed_bounds"
MULTISEED_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "multiseed"

def newest_trace(root):
    matches = sorted(root.glob("phase2/**/trace.npz"), key=lambda p: p.stat().st_mtime)
    return matches[-1] if matches else None

def frozen_dir(seed):
    candidates = [MULTISEED_ROOT / f"trace_seed_{seed}"]
    if int(seed) == int(SEED):
        candidates.insert(0, RESULTS_ROOT)
    for root in candidates:
        if (root / "phase4" / "hybrid_quality_by_policy.csv").is_file():
            return root
    return candidates[-1]

def assert_trace_parity(seed):
    frozen = newest_trace(frozen_dir(seed))
    bounds = newest_trace(BOUNDS_ROOT / f"trace_seed_{seed}")
    if frozen is None or bounds is None:
        print(f"  seed {seed}: trace missing (frozen={frozen}, bounds={bounds}) - skipping parity check")
        return
    a, b = np.load(frozen), np.load(bounds)
    for key in ("noise_std", "available", "faulted", "class_code", "thermal_zone", "fault_onset"):
        if not np.array_equal(a[key], b[key]):
            raise AssertionError(
                f"Trace mismatch for seed {seed}, array {key}: the bounds run is NOT "
                "consistent with the frozen campaign. Check for config overrides."
            )
    print(f"  seed {seed}: trace parity OK ({bounds.name} == frozen)")

frames = []
for seed_dir in sorted(BOUNDS_ROOT.glob("trace_seed_*")):
    seed = int(seed_dir.name.split("_")[-1])
    bounds_csv = seed_dir / "phase4" / "hybrid_quality_by_policy.csv"
    frozen_csv = frozen_dir(seed) / "phase4" / "hybrid_quality_by_policy.csv"
    if not bounds_csv.is_file():
        print(f"  seed {seed}: no bounds Phase-4 CSV yet - skipping")
        continue
    if not frozen_csv.is_file():
        print(f"  seed {seed}: frozen Phase-4 CSV not found at {frozen_csv} - skipping")
        continue
    assert_trace_parity(seed)
    frozen_df = pd.read_csv(frozen_csv)
    bounds_df = pd.read_csv(bounds_csv)
    common = [c for c in bounds_df.columns if c in frozen_df.columns]
    merged = pd.concat([frozen_df[common], bounds_df[common]], ignore_index=True)
    merged["trace_seed"] = seed
    frames.append(merged)

if not frames:
    raise RuntimeError("No completed bounds seeds found under " + str(BOUNDS_ROOT))

data = pd.concat(frames, ignore_index=True)

# Per-seed mean delta_nll_tile per policy (the quantity placement competes on).
per_seed = (
    data.groupby(["trace_seed", "policy"])["delta_nll_tile"].mean().unstack("policy")
)
display(per_seed.round(4))

summary = per_seed.mean()
span = summary["adversarial"] - summary["oracle_clairvoyant"]
capture_full = (summary["adversarial"] - summary["static_sensitivity"]) / span
capture_vs_random = (
    (summary["random"] - summary["static_sensitivity"])
    / (summary["random"] - summary["oracle_clairvoyant"])
)
oracle_gap = summary["static_sensitivity"] - summary["oracle_clairvoyant"]
print(f"Cross-seed mean delta_nll_tile (nats): " + ", ".join(
    f"{name}={summary[name]:.4f}" for name in summary.index))
print(f"Placement span (adversarial - oracle): {span:.4f} nats")
print(f"static_sensitivity captures {capture_full:.1%} of the full worst-to-oracle span")
print(f"static_sensitivity captures {capture_vs_random:.1%} of the random-to-oracle headroom")
print(f"Residual gap to the clairvoyant oracle: {oracle_gap:.4f} nats")

out_csv = BOUNDS_ROOT / "bounds_span_summary.csv"
per_seed.to_csv(out_csv)
print("Per-seed table saved to:", out_csv)


In [ ]:
#@title 8f. Phase-2 robustness sweep (one variant per run)
#@markdown One-factor-at-a-time variants of the frozen Phase-2 scenario, run on
#@markdown the SEED=42 trace with the reduced Phase-4 grid baked into the
#@markdown variant configs (t in {0, 119}, 2 realizations, 3 policies). Run
#@markdown Cells 1-7 first, then this cell once per variant. Reuses Phase 0/1;
#@markdown rerunning after a preemption resumes Phase 4.
SWEEP_VARIANT = "faults_low"  #@param ["faults_low", "faults_high", "drift_high", "spread_low", "spread_high", "thermal_low"]

import subprocess, sys, yaml

assert USE_HWA, "The sweep extends the HWA study; set USE_HWA = True."

sweep_base = PROJECT_DIR / "configs" / "phase2_sweep" / f"{SWEEP_VARIANT}.yaml"
if not sweep_base.is_file():
    raise FileNotFoundError(f"Sweep variant config not in the checkout: {sweep_base}")
sweep_config = yaml.safe_load(sweep_base.read_text(encoding="utf-8"))
sweep_config.setdefault("experiment", {})
sweep_config["experiment"]["seed"] = int(SEED)
sweep_config["experiment"]["placement_seed"] = int(SEED)
sweep_config.setdefault("model", {})
sweep_config["model"]["device"] = RUNTIME_DEVICE
sweep_config["model"]["checkpoint"] = str(HWA_CHECKPOINT_DIR)

SWEEP_CONFIG = CONFIG_ROOT / f"gpt2_sweep_{SWEEP_VARIANT}_{RUN_MODE}_{MODEL_VARIANT}.yaml"
SWEEP_CONFIG.write_text(yaml.safe_dump(sweep_config, sort_keys=False), encoding="utf-8")
SWEEP_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "sweep" / SWEEP_VARIANT

command = [
    sys.executable, "-u", "scripts/run_multiseed_pipeline.py",
    "--config", str(SWEEP_CONFIG),
    "--phase1", str(PHASE1_PATH),
    "--trace-seeds", str(SEED),
    "--vary-placement-seed",
    "--output-root", str(SWEEP_ROOT),
]
print("+", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)
print("Sweep variant complete:", SWEEP_VARIANT)


In [ ]:
#@title 8g. Sweep summary table (run after the variants you want)
#@markdown Collects every completed sweep variant plus the frozen scenario and
#@markdown tabulates the paired improvement of static_sensitivity over each
#@markdown baseline. The claim to check: the improvement stays positive in every
#@markdown variant and scales with tile-quality spread.
import pandas as pd
from IPython.display import display

SWEEP_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "sweep"
rows = []

def add_paired_summary(label, csv_path):
    if not csv_path.is_file():
        return False
    df = pd.read_csv(csv_path)
    df = df[df["method_policy"] == "static_sensitivity"]
    for _, row in df.iterrows():
        rows.append({
            "variant": label,
            "baseline": row["baseline_policy"],
            "mean_improvement_nats": row["mean_nll_improvement"],
            "win_fraction": row["win_fraction"],
            "paired_samples": row["paired_samples"],
        })
    return True

# Frozen scenario (full grid, 15 conditions) as the reference row; the seed-42
# run may live in the seed tree or under multiseed/trace_seed_<SEED>.
MULTISEED_ROOT = STORAGE_ROOT / "results" / RUN_MODE / MODEL_VARIANT / "multiseed"
for _frozen_root in (RESULTS_ROOT, MULTISEED_ROOT / f"trace_seed_{SEED}"):
    if add_paired_summary("frozen (paper)", _frozen_root / "phase4" / "paired_policy_summary.csv"):
        break
for variant_dir in sorted(SWEEP_ROOT.glob("*/trace_seed_*")):
    label = variant_dir.parent.name
    if not add_paired_summary(label, variant_dir / "phase4" / "paired_policy_summary.csv"):
        print("No Phase-4 summary yet for variant:", label)

if not rows:
    raise RuntimeError("No completed sweep variants found under " + str(SWEEP_ROOT))

table = pd.DataFrame(rows).pivot_table(
    index="variant", columns="baseline",
    values=["mean_improvement_nats", "win_fraction"],
)
display(table.round(4))
out_csv = SWEEP_ROOT / "sweep_summary.csv"
table.to_csv(out_csv)
print("Saved to:", out_csv)
print("NOTE: frozen-row improvements average 15 full-grid conditions; sweep rows")
print("average 4 reduced-grid conditions (t in {0,119} x 2 realizations), so they")
print("emphasize end-of-life more than the frozen row does.")


In [ ]:
#@title 8h. Leave-one-out check of the Phase-1 profile (all 49 projections analog)
#@markdown Reviewer-requested rank-stability check: Phase 1 measures each
#@markdown projection with the other 48 digital, whereas deployment noises all
#@markdown 49 at once. This cell re-measures every projection as
#@markdown s_loo(p) = E[NLL(all analog, noisy) - NLL(all analog, noisy, p restored)]
#@markdown with exactly the Phase-1 noise fields and reports Spearman/Kendall
#@markdown rank agreement with the isolated profile. Run Cells 1-7 first
#@markdown (SEED = 42, USE_HWA = True, every RUN_PHASE... = 0). Cost is about
#@markdown one Phase-1 run (501 calibration passes on the GPU). `nominal`
#@markdown keeps p analog but noise-free (isolates p's noise exactly as Eq. (1)
#@markdown of the paper); `digital` swaps p back to its floating-point module.
LOO_RESTORE_MODE = "nominal"  #@param ["nominal", "digital"]
LOO_MAX_BATCHES = 0  #@param {type:"integer"}

import subprocess, sys

assert USE_HWA, "The leave-one-out check uses the HWA checkpoint and its Phase-1 profile; set USE_HWA = True."
assert PHASE1_PATH is not None and PHASE1_PATH.is_file(), "Resolve the Phase-1 artifact in Cell 7 first."

command = [
    sys.executable, "-u", "experiments/phase1_sensitivity/run_leave_one_out.py",
    "--config", str(COLAB_CONFIG),
    "--phase1", str(PHASE1_PATH),
    "--restore-mode", LOO_RESTORE_MODE,
]
if int(LOO_MAX_BATCHES) > 0:
    command += ["--max-batches", str(int(LOO_MAX_BATCHES))]
print("+", " ".join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)
print("Leave-one-out check complete; see <phase1 output root>/leave_one_out/ on Drive.")


In [ ]:
#@title 9. Inspect current artifacts
import json
import pandas as pd
from IPython.display import display

def show_csv(path, columns=None):
    if not path.is_file():
        print("Not found:", path)
        return
    print(path)
    frame = pd.read_csv(path)
    if columns:
        selected = [column for column in columns if column in frame.columns]
        if selected:
            frame = frame[selected]
    display(frame)

phase0_metadata = RESULTS_ROOT / "phase0" / "hwa_metadata.json"
if phase0_metadata.is_file():
    payload = json.loads(phase0_metadata.read_text(encoding="utf-8"))
    print("Phase 0 final loss:", payload.get("final_loss"))
    history = payload.get("eval_history", [])
    if history:
        display(pd.DataFrame(history))
else:
    print("No Phase 0 metadata yet.")

rankings = sorted(RESULTS_ROOT.glob("phase1/*_ranking.csv"))
if rankings:
    show_csv(rankings[-1])

nominal = RESULTS_ROOT / "phase4" / "nominal_reference.json"
if nominal.is_file():
    print(json.dumps(json.loads(nominal.read_text(encoding="utf-8")), indent=2))

for name in [
    "phase3_summary.csv",
    "hybrid_quality_summary.csv",
    "paired_policy_summary.csv",
]:
    matches = sorted(
        RESULTS_ROOT.rglob(name),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if matches:
        print("\n", name)
        show_csv(matches[0])


## Usage

1. In Cell 1, choose `USE_HWA` (hardware-aware checkpoint) or the PTQ
   contrast, and set each `RUN_PHASE...` value to `1` or `0`.
2. In Cell 5, put only the values you want to change in
   `COLAB_CONFIG_OVERRIDES_YAML`; the repository configuration supplies
   everything else.
3. Run Cells 1-7 after starting a new runtime.
4. Run Cell 8 to execute every enabled phase in pipeline order.
5. Run Cell 9 to inspect outputs (Phase 0 eval history, sensitivity
   ranking, nominal reference, quality and paired
   summaries).

Phase 0 resumes automatically from its newest step checkpoint if the
session was preempted; Phase 4 resumes from its per-timestep partial
checkpoint the same way -- rerun Cell 8 with the same flags. With
`RUN_PHASE0 = 0` and `USE_HWA = True`, the Phase 0 checkpoint must already
exist under the results directory from a previous session.

When a prerequisite phase is disabled, the notebook uses the newest
matching artifact already stored under the selected results directory. Set
an explicit artifact path in Cell 1 to override.

HWA and PTQ artifacts live in separate trees:
`results/<mode>/hwa/seed_<seed>` versus `results/<mode>/ptq/seed_<seed>`,
so the two variants can never overwrite each other. For independent
hardware traces use `scripts/run_multiseed_pipeline.py` (reuses the Phase-1
profile across trace seeds), then
`scripts/aggregate_multiseed.py` for the cross-trace paper table.

### Bounds and sweep (additive experiments)

- **Cell 8d** — run once per `BOUNDS_TRACE_SEED` (41–45). Evaluates the
  `adversarial` (worst-case) and `oracle_clairvoyant` (trajectory-RMS upper
  bound) placements under the identical noise contract; results land in
  `multiseed_bounds/` and stay exactly paired with the frozen campaign.
- **Cell 8e** — after any number of bounds seeds: verifies each regenerated
  trace is identical to the frozen one, then reports the placement span and the
  headroom fraction `static_sensitivity` captures.
- **Cell 8f** — run once per `SWEEP_VARIANT` (six one-factor Phase-2 variants,
  SEED 42, reduced Phase-4 grid); results land in `sweep/<variant>/`.
- **Cell 8g** — tabulates paired improvements across completed variants.

All four cells require Cells 1–7 to have been run first in the same session
with `SEED = 42`, `USE_HWA = True`, and every `RUN_PHASE... = 0`.
- Cell 8h runs the leave-one-out rank-stability check of the Phase-1 profile (all 49 projections analog, one restored at a time; about one Phase-1 run of GPU time) and writes `leave_one_out/` next to the Phase-1 artifact.